In [ ]:
# Распаковка датасета
!unzip /content/dataset.zip -d /content

In [ ]:
# Установка необходимых библиотек
import os
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from PIL import Image
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from skimage.metrics import structural_similarity as ssim
import zipfile
import io

In [ ]:
# Проверка GPU
print("Доступные GPU:", tf.config.list_physical_devices('GPU'))
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Memory growth включен для всех GPU")
    except RuntimeError as e:
        print(e)
print("Используемое устройство:", tf.config.get_visible_devices())

In [ ]:
# Пути к данным
input_dir = '/content/dataset/input'
target_dir = '/content/dataset/target'
input_paths = sorted(glob(os.path.join(input_dir, '*.png')))
target_paths = sorted(glob(os.path.join(target_dir, '*.png')))
print(f'Примеров в input: {len(input_paths)}, в target: {len(target_paths)}')

# Визуализация примера
idx = 0
img_input = np.array(Image.open(input_paths[idx]).convert('L'))
img_target = np.array(Image.open(target_paths[idx]).convert('L'))
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.title('Зашумленное')
plt.imshow(img_input, cmap='gray')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.title('Очищенное')
plt.imshow(img_target, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
# Подготовка датасета
IMG_SIZE = (512, 512)
BATCH_SIZE = 4

def create_dataset(paths, batch_size=8):
    def load_and_preprocess(path):
        # Загружаем изображение
        img = tf.io.read_file(path)
        img = tf.image.decode_png(img, channels=1)  # grayscale
        # Преобразуем в float32 и нормализуем в диапазон [0, 1]
        img = tf.cast(img, tf.float32) / 255.0
        img = tf.clip_by_value(img, 0.0, 1.0)
        return img

    dataset = tf.data.Dataset.from_tensor_slices(paths)
    dataset = dataset.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset


In [ ]:
# Разделение на train/test
train_input_paths, test_input_paths, train_target_paths, test_target_paths = train_test_split(
    input_paths, target_paths, test_size=0.1, random_state=42
)

# Создаем датасеты
BATCH_SIZE = 8
train_input_ds = create_dataset(train_input_paths, BATCH_SIZE)
train_target_ds = create_dataset(train_target_paths, BATCH_SIZE)
test_input_ds = create_dataset(test_input_paths, BATCH_SIZE)
test_target_ds = create_dataset(test_target_paths, BATCH_SIZE)

# Объединяем входы и таргеты
train_ds = tf.data.Dataset.zip((train_input_ds, train_target_ds))
test_ds = tf.data.Dataset.zip((test_input_ds, test_target_ds))

print(f"Количество батчей в train_ds: {len(list(train_ds))}")
print(f"Количество батчей в test_ds: {len(list(test_ds))}")

In [ ]:
# U-Net модель
def create_unet(input_shape=(512, 512, 1)):
    inputs = keras.Input(shape=input_shape)

    # Encoder
    conv1 = layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)
    conv1 = layers.BatchNormalization()(conv1)
    conv1 = layers.Conv2D(32, 3, activation='relu', padding='same')(conv1)
    conv1 = layers.BatchNormalization()(conv1)
    pool1 = layers.MaxPooling2D(pool_size=(2, 2))(conv1)

    conv2 = layers.Conv2D(64, 3, activation='relu', padding='same')(pool1)
    conv2 = layers.BatchNormalization()(conv2)
    conv2 = layers.Conv2D(64, 3, activation='relu', padding='same')(conv2)
    conv2 = layers.BatchNormalization()(conv2)
    pool2 = layers.MaxPooling2D(pool_size=(2, 2))(conv2)

    conv3 = layers.Conv2D(128, 3, activation='relu', padding='same')(pool2)
    conv3 = layers.BatchNormalization()(conv3)
    conv3 = layers.Conv2D(128, 3, activation='relu', padding='same')(conv3)
    conv3 = layers.BatchNormalization()(conv3)
    pool3 = layers.MaxPooling2D(pool_size=(2, 2))(conv3)

    # Bottleneck
    conv4 = layers.Conv2D(256, 3, activation='relu', padding='same')(pool3)
    conv4 = layers.BatchNormalization()(conv4)
    conv4 = layers.Conv2D(256, 3, activation='relu', padding='same')(conv4)
    conv4 = layers.BatchNormalization()(conv4)

    # Decoder
    up3 = layers.Conv2DTranspose(128, 2, strides=(2, 2), padding='same')(conv4)
    concat3 = layers.Concatenate()([conv3, up3])
    conv3_up = layers.Conv2D(128, 3, activation='relu', padding='same')(concat3)
    conv3_up = layers.BatchNormalization()(conv3_up)
    conv3_up = layers.Conv2D(128, 3, activation='relu', padding='same')(conv3_up)
    conv3_up = layers.BatchNormalization()(conv3_up)

    up2 = layers.Conv2DTranspose(64, 2, strides=(2, 2), padding='same')(conv3_up)
    concat2 = layers.Concatenate()([conv2, up2])
    conv2_up = layers.Conv2D(64, 3, activation='relu', padding='same')(concat2)
    conv2_up = layers.BatchNormalization()(conv2_up)
    conv2_up = layers.Conv2D(64, 3, activation='relu', padding='same')(conv2_up)
    conv2_up = layers.BatchNormalization()(conv2_up)

    up1 = layers.Conv2DTranspose(32, 2, strides=(2, 2), padding='same')(conv2_up)
    concat1 = layers.Concatenate()([conv1, up1])
    conv1_up = layers.Conv2D(32, 3, activation='relu', padding='same')(concat1)
    conv1_up = layers.BatchNormalization()(conv1_up)
    conv1_up = layers.Conv2D(32, 3, activation='relu', padding='same')(conv1_up)
    conv1_up = layers.BatchNormalization()(conv1_up)

    outputs = layers.Conv2D(1, 1, activation='sigmoid')(conv1_up)
    model = keras.Model(inputs=[inputs], outputs=[outputs])
    return model

# Функция потерь
def photon_loss(y_true, y_pred):
    mse_loss = tf.reduce_mean(tf.square(y_true - y_pred))
    total_true = tf.reduce_sum(y_true)
    total_pred = tf.reduce_sum(y_pred)
    photon_loss = tf.abs(total_true - total_pred) / (total_true + 1e-7)
    return mse_loss + 0.5 * photon_loss

def photon_metrics(y_true, y_pred):
    total_true = tf.reduce_sum(y_true)
    total_pred = tf.reduce_sum(y_pred)
    return tf.abs(total_true - total_pred) / (total_true + 1e-7)

# Создание модели
model = create_unet()
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=1e-5),
    loss=photon_loss,
    metrics=['mae', photon_metrics]
)
model.summary()

# Callbacks
callbacks = [
    keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.2, patience=5),
    keras.callbacks.ModelCheckpoint('best_model.keras', save_best_only=True, monitor='val_loss')
]

In [ ]:
# Обучение
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=100,
    callbacks=callbacks
)

In [ ]:
# Графики потерь и метрик
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['mae'], label='train')
plt.plot(history.history['val_mae'], label='val')
plt.title('MAE')
plt.legend()
plt.show()

# Функция для визуализации и сохранения результатов
def show_and_save_results(idx=0, archive=None):
    test_images = None
    test_labels = None

    # Получаем нужный батч из тестового датасета
    for images, labels in test_ds.skip(idx // BATCH_SIZE).take(1):
        test_images = images
        test_labels = labels

    if test_images is None or test_labels is None:
        print(f"Ошибка: не удалось получить изображение с индексом {idx}")
        return

    # Получаем конкретное изображение из батча
    batch_idx = idx % BATCH_SIZE
    input_img = tf.expand_dims(test_images[batch_idx], 0)  # Добавляем размерность батча
    true_img = tf.expand_dims(test_labels[batch_idx], 0)
    # Делаем предсказание
    pred_img = model.predict(input_img, verbose=0)

    # Нормализуем предсказания в диапазон [0, 1]
    pred_min = np.min(pred_img)
    pred_max = np.max(pred_img)
    pred_img = (pred_img - pred_min) / (pred_max - pred_min + 1e-7)

    # Преобразуем тензоры в numpy массивы
    true_img_np = true_img.numpy()[0, ..., 0]  # true_img — это tf.Tensor, .numpy() нужен
    pred_img_np = pred_img[0, ..., 0]  # pred_img — это numpy.ndarray, .numpy() не нужен

    # Визуализация
    plt.figure(figsize=(15, 8))
    plt.subplot(2, 3, 1)
    plt.title('Зашумленное')
    plt.imshow(input_img[0, ..., 0], cmap='gray', vmin=0, vmax=1)
    plt.axis('off')

    plt.subplot(2, 3, 2)
    plt.title(f'Предсказание\nmin={pred_min:.3f}, max={pred_max:.3f}')
    plt.imshow(pred_img[0, ..., 0], cmap='gray', vmin=0, vmax=1)
    plt.axis('off')

    plt.subplot(2, 3, 3)
    plt.title('Истинное')
    plt.imshow(true_img[0, ..., 0], cmap='gray', vmin=0, vmax=1)
    plt.axis('off')

    plt.subplot(2, 3, 4)
    plt.title('Гистограмма зашумленного')
    plt.hist(input_img.numpy()[0, ..., 0].flatten(), bins=50, range=(0, 1))
    plt.grid()

    plt.subplot(2, 3, 5)
    plt.title('Гистограмма предсказания')
    plt.hist(pred_img[0, ..., 0].flatten(), bins=50, range=(0, 1))
    plt.grid()

    plt.subplot(2, 3, 6)
    plt.title('Гистограмма истинного')
    plt.hist(true_img.numpy()[0, ..., 0].flatten(), bins=50, range=(0, 1))
    plt.grid()

    plt.tight_layout()
    plt.show()

    # Статистика
    print("\nСтатистика значений:")
    print(f"Зашумленное - min: {np.min(input_img):.4f}, max: {np.max(input_img):.4f}, mean: {np.mean(input_img):.4f}")
    print(f"Предсказание - min: {np.min(pred_img):.4f}, max: {np.max(pred_img):.4f}, mean: {np.mean(pred_img):.4f}")
    print(f"Истинное - min: {np.min(true_img):.4f}, max: {np.max(true_img):.4f}, mean: {np.mean(true_img):.4f}")

    # Приведение к одинаковому размеру, если нужно
    if true_img_np.shape != pred_img_np.shape:
        target_height, target_width = true_img_np.shape
        pred_img_np = tf.image.resize(
            tf.convert_to_tensor(pred_img_np)[np.newaxis, ..., np.newaxis],
            [target_height, target_width]
        ).numpy()[0, ..., 0]

    # Вычисление метрик
    ssim_score = ssim(true_img_np, pred_img_np, data_range=1.0)
    mae_score = np.mean(np.abs(true_img_np - pred_img_np))
    print(f"\nМетрики для изображения {idx}:")
    print(f"SSIM: {ssim_score:.4f}")
    print(f"MAE: {mae_score:.4f}")

    # Сохранение в ZIP
    if archive is not None:
        input_array = (input_img[0, ..., 0].numpy() * 255).astype('uint8')
        pred_array = (pred_img_np * 255).astype('uint8')
        true_array = (true_img_np * 255).astype('uint8')

        input_pil = Image.fromarray(input_array, mode='L')
        pred_pil = Image.fromarray(pred_array, mode='L')
        true_pil = Image.fromarray(true_array, mode='L')

        input_buffer = io.BytesIO()
        pred_buffer = io.BytesIO()
        true_buffer = io.BytesIO()

        input_pil.save(input_buffer, format='PNG')
        pred_pil.save(pred_buffer, format='PNG')
        true_pil.save(true_buffer, format='PNG')

        archive.writestr(f'test_{idx:03d}_input.png', input_buffer.getvalue())
        archive.writestr(f'test_{idx:03d}_predicted.png', pred_buffer.getvalue())
        archive.writestr(f'test_{idx:03d}_true.png', true_buffer.getvalue())

        input_buffer.close()
        pred_buffer.close()
        true_buffer.close()


# Сохранение результатов
with zipfile.ZipFile('predictions.zip', 'w') as archive:
    for i in range(5):
        show_and_save_results(i, archive)
    print("\nРезультаты сохранены в predictions.zip")